# ReefScan AI — Finalize Model & Inference

This notebook prepares the final trained model for inference.

Tasks:
1. Load the final model
2. Define the final label mapping
3. Save the per-label decision thresholds
4. Build an inference function
5. Test inference using new coral images

In [1]:
from pathlib import Path
import json
import numpy as np
import tensorflow as tf

PROJECT_DIR = Path("..")
MODEL_DIR = PROJECT_DIR / "models"
NEW_IMAGES_DIR = PROJECT_DIR / "new_images"

MODEL_PATH = (
    MODEL_DIR /
    "mobilenetv2_multilabel_finetuned.keras"
)

MODEL_LABELS = [1, 2, 3, 4, 5, 6, 8]

LABEL_NAMES = [
    "Healthy coral",
    "Compromised coral",
    "Dead coral",
    "Rubble",
    "Competition",
    "Disease",
    "Physical issues"
]

FINAL_THRESHOLDS = {
    "Healthy coral": 0.45,
    "Compromised coral": 0.35,
    "Dead coral": 0.25,
    "Rubble": 0.40,
    "Competition": 0.30,
    "Disease": 0.30,
    "Physical issues": 0.35
}

IMG_SIZE = 224

print("Model path:", MODEL_PATH)
print("Model exists:", MODEL_PATH.exists())

Model path: ..\models\mobilenetv2_multilabel_finetuned.keras
Model exists: True


In [2]:
model = tf.keras.models.load_model(
    MODEL_PATH,
    compile=False
)

model.summary()

Model: "MobileNetV2_MultiLabel"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,587,719 (9.87 MB)

 Trainable params: 2,191,175 (8.36 MB)

 Non-trainable params: 396,544 (1.51 MB)

In [3]:
THRESHOLD_PATH = MODEL_DIR / "reefscan_thresholds.json"

with open(THRESHOLD_PATH, "w", encoding="utf-8") as f:
    json.dump(
        FINAL_THRESHOLDS,
        f,
        indent=4
    )

print("Threshold file saved to:", THRESHOLD_PATH)

Threshold file saved to: ..\models\reefscan_thresholds.json


In [4]:
with open(THRESHOLD_PATH, "r", encoding="utf-8") as f:
    print(json.load(f))

{'Healthy coral': 0.45, 'Compromised coral': 0.35, 'Dead coral': 0.25, 'Rubble': 0.4, 'Competition': 0.3, 'Disease': 0.3, 'Physical issues': 0.35}


In [5]:
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from PIL import Image


def predict_image(image_path):
    # Load image
    image = Image.open(image_path).convert("RGB")

    # Resize
    image = image.resize((IMG_SIZE, IMG_SIZE))

    # Convert to NumPy
    image_array = np.array(image, dtype=np.float32)

    # Add batch dimension
    image_array = np.expand_dims(image_array, axis=0)

    # MobileNetV2 preprocessing
    image_array = preprocess_input(image_array)

    # Predict probabilities
    probabilities = model.predict(
        image_array,
        verbose=0
    )[0]

    # Apply per-label thresholds
    predictions = {}

    for label_name, probability in zip(
        LABEL_NAMES,
        probabilities
    ):
        threshold = FINAL_THRESHOLDS[label_name]

        predictions[label_name] = {
            "probability": float(probability),
            "threshold": threshold,
            "predicted": bool(probability >= threshold)
        }

    return predictions

In [10]:
test_image = NEW_IMAGES_DIR / "coral3.jpg"

print("Testing:", test_image)
print("Exists :", test_image.exists())

predictions = predict_image(test_image)

for label_name, result in predictions.items():
    print(
        f"{label_name:20s} "
        f"prob={result['probability']:.3f} "
        f"threshold={result['threshold']:.2f} "
        f"predicted={result['predicted']}"
    )

Testing: ..\new_images\coral3.jpg
Exists : True
Healthy coral        prob=0.835 threshold=0.45 predicted=True
Compromised coral    prob=0.064 threshold=0.35 predicted=False
Dead coral           prob=0.587 threshold=0.25 predicted=True
Rubble               prob=0.021 threshold=0.40 predicted=False
Competition          prob=0.397 threshold=0.30 predicted=True
Disease              prob=0.005 threshold=0.30 predicted=False
Physical issues      prob=0.020 threshold=0.35 predicted=False


In [11]:
detected_labels = [
    label_name
    for label_name, result in predictions.items()
    if result["predicted"]
]

print("Detected labels:")

for label in detected_labels:
    print("-", label)

if not detected_labels:
    print("- No target condition detected")

Detected labels:
- Healthy coral
- Dead coral
- Competition


In [14]:
import pandas as pd
import ast

TEST_PATH = PROJECT_DIR / "data" / "processed" / "test.csv"

test_df = pd.read_csv(TEST_PATH)

test_df["labels"] = test_df["labels"].apply(ast.literal_eval)

print("Test dataset:", test_df.shape)
display(test_df.head())

Test dataset: (3162, 13)


,patchid,label,image_path,valid,width,height,brightness,contrast,blur_score,quality_issue,quality_issues,is_quality_issue,labels
0,20230412_0001_15,"[1, 2]",..\data\raw\coral_images\20230421_preliminary_...,True,512,512,100.537514,30.675453,134.715802,normal,[],False,"[1, 2]"
1,20230412_0001_21,"[1, 2]",..\data\raw\coral_images\20230421_preliminary_...,True,512,512,104.583557,25.093023,114.474743,normal,[],False,"[1, 2]"
2,20230412_0001_25,[2],..\data\raw\coral_images\20230421_preliminary_...,True,512,512,134.809101,29.261308,64.825077,normal,[],False,[2]
3,20230412_0001_27,[2],..\data\raw\coral_images\20230421_preliminary_...,True,512,512,92.843243,38.804045,81.100323,normal,[],False,[2]
4,20230412_0001_32,[2],..\data\raw\coral_images\20230421_preliminary_...,True,512,512,109.576752,71.905367,73.337259,normal,[],False,[2]


In [15]:
test_sample = test_df.iloc[0]

print("Patch ID:", test_sample["patchid"])
print("Ground truth:", test_sample["labels"])
print("Image:", test_sample["image_path"])

Patch ID: 20230412_0001_15
Ground truth: [1, 2]
Image: ..\data\raw\coral_images\20230421_preliminary_survey\clipped\20230412_0001_15.jpg


In [16]:
predictions = predict_image(
    test_sample["image_path"]
)

for label_name, result in predictions.items():
    print(
        f"{label_name:20s} "
        f"prob={result['probability']:.3f} "
        f"threshold={result['threshold']:.2f} "
        f"predicted={result['predicted']}"
    )

Healthy coral        prob=0.987 threshold=0.45 predicted=True
Compromised coral    prob=0.195 threshold=0.35 predicted=False
Dead coral           prob=0.104 threshold=0.25 predicted=False
Rubble               prob=0.005 threshold=0.40 predicted=False
Competition          prob=0.003 threshold=0.30 predicted=False
Disease              prob=0.025 threshold=0.30 predicted=False
Physical issues      prob=0.159 threshold=0.35 predicted=False


In [17]:
test_samples = test_df.sample(
    12,
    random_state=42
)

for _, row in test_samples.iterrows():

    predictions = predict_image(row["image_path"])

    predicted_labels = [
        MODEL_LABELS[i]
        for i, label_name in enumerate(LABEL_NAMES)
        if predictions[label_name]["predicted"]
    ]

    print("Patch:", row["patchid"])
    print("True :", row["labels"])
    print("Pred :", predicted_labels)
    print("-" * 60)

Patch: 20230412_0027_5
True : [1, 2]
Pred : [1, 2]
------------------------------------------------------------
Patch: HNM_0031_00_20230928_0018_4
True : [2, 5, 6]
Pred : [1, 3, 5]
------------------------------------------------------------
Patch: ALK_0030_11_20230927_0015_22
True : [1, 2, 8]
Pred : [1, 2, 3, 8]
------------------------------------------------------------
Patch: HNM_0031_00_20230928_0092_18
True : [3]
Pred : [2]
------------------------------------------------------------
Patch: ALK_0030_11_20230927_0083_30
True : [1, 3, 5]
Pred : [1, 3, 4]
------------------------------------------------------------
Patch: HNM_0031_00_20230928_0091_3
True : [3]
Pred : [3]
------------------------------------------------------------
Patch: ALK_0030_11_20230927_0082_4
True : [1, 3, 5]
Pred : [1, 3, 4]
------------------------------------------------------------
Patch: ALK_0030_11_20230927_0043_29
True : [1, 2, 6]
Pred : [2, 6]
-----------------------------------------------------------

In [18]:
id_to_name = dict(zip(MODEL_LABELS, LABEL_NAMES))

for _, row in test_samples.iterrows():

    predictions = predict_image(row["image_path"])

    true_names = [
        id_to_name[label]
        for label in row["labels"]
    ]

    predicted_names = [
        label_name
        for label_name, result in predictions.items()
        if result["predicted"]
    ]

    print("Patch:", row["patchid"])
    print("True :", true_names)
    print("Pred :", predicted_names)
    print("-" * 60)

Patch: 20230412_0027_5
True : ['Healthy coral', 'Compromised coral']
Pred : ['Healthy coral', 'Compromised coral']
------------------------------------------------------------
Patch: HNM_0031_00_20230928_0018_4
True : ['Compromised coral', 'Competition', 'Disease']
Pred : ['Healthy coral', 'Dead coral', 'Competition']
------------------------------------------------------------
Patch: ALK_0030_11_20230927_0015_22
True : ['Healthy coral', 'Compromised coral', 'Physical issues']
Pred : ['Healthy coral', 'Compromised coral', 'Dead coral', 'Physical issues']
------------------------------------------------------------
Patch: HNM_0031_00_20230928_0092_18
True : ['Dead coral']
Pred : ['Compromised coral']
------------------------------------------------------------
Patch: ALK_0030_11_20230927_0083_30
True : ['Healthy coral', 'Dead coral', 'Competition']
Pred : ['Healthy coral', 'Dead coral', 'Rubble']
------------------------------------------------------------
Patch: HNM_0031_00_20230928_00